# Mini-MILP: comunidades microbianas mínimas (ejemplo de juguete)

**Objetivo:** mostrar cómo una misma formulación puede seleccionar microorganismos para transformar un sustrato en un producto, **sin programar una ruta particular en el algoritmo**.

Este notebook es **didáctico y autocontenido**: utiliza reacciones hipotéticas simplificadas, balances de un *pool* compartido y selección binaria de especies. **No es SteadyCom, FBA basado en GEMs ni una validación biológica.** Para resultados reales del proyecto se necesitan `E2/src/comunidad.py`, `E2/src/milp.py`, modelos AGORA2 y controles de crecimiento.

Solo requiere `numpy`, `pandas` y `scipy` (incluidos en la mayoría de entornos científicos).

## 1. Formulación del problema

Cada reacción candidata $r$ transforma ciertos metabolitos, tiene una capacidad máxima $U_r$ y pertenece a una especie $k$.

- $v_r \geq 0$: flujo de la reacción.
- $y_k \in \{0,1\}$: selección de la especie.
- $u_s$: entrada del sustrato desde el medio.
- $e_p$: salida del producto objetivo.

$$\min \sum_k y_k$$

sujeto a balances de masa de los metabolitos, disponibilidad del medio, producción mínima $e_p\geq\theta$ y acoplamiento $0\leq v_r\leq U_r y_{k(r)}$.

**Supuesto didáctico importante:** se modela un único pool compartido; no se representan compartimentos intracelulares, abundancias, crecimiento o limitaciones termodinámicas.

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import milp, Bounds, LinearConstraint

print("Notebook listo (SciPy MILP disponible).")

Notebook listo (SciPy MILP disponible).


## 2. Solucionador general

La función recibe **datos**, no nombres específicos de levodopa, dopamina o bacterias. Puedes reutilizarla con otras rutas definidas por este formato:

```python
{"especie": "A", "nombre": "R1", "stoich": {"S": -1, "I": 1}, "capacidad": 1.0}
```

Signo negativo: reactivo consumido. Positivo: producto generado.

In [ ]:
def comunidad_minima(reacciones, medio, producto, theta, ancla=None):
    """MILP didáctico: elegir mínimo número de especies para producir 'producto'.

    reacciones : lista de dicts con especie, nombre, stoich, capacidad
    medio     : dict de metabolito -> captación máxima externa
    theta     : exportación mínima del producto
    ancla     : especie obligatoria (o None)
    """
    especies = list(dict.fromkeys(r["especie"] for r in reacciones))
    if ancla is not None and ancla not in especies:
        raise ValueError("El ancla debe figurar entre las especies candidatas")
    if theta <= 0 or any(v < 0 for v in medio.values()):
        raise ValueError("theta debe ser positiva y las captaciones no negativas")

    metabolitos = sorted(set(medio) | {producto} |
                         {m for r in reacciones for m in r["stoich"]})
    if producto in medio:
        raise ValueError("El producto objetivo no debe estar disponible desde el ambiente")
    if any(r["capacidad"] <= 0 for r in reacciones):
        raise ValueError("Todas las capacidades deben ser positivas")

    # Índices de variables: [flujos de reacciones, captaciones, exportación, binarias]
    nr = len(reacciones)
    suministros = list(medio)
    nu = len(suministros)
    idx_u = {m: nr + i for i, m in enumerate(suministros)}
    idx_e = nr + nu
    idx_y = {k: nr + nu + 1 + i for i, k in enumerate(especies)}
    n = nr + nu + 1 + len(especies)

    objetivo = np.zeros(n)
    for k, i in idx_y.items():
        objetivo[i] = 1.0

    lb = np.zeros(n)
    ub = np.full(n, np.inf)
    for j, r in enumerate(reacciones):
        ub[j] = float(r["capacidad"])
    for m, i in idx_u.items():
        ub[i] = float(medio[m])
    lb[idx_e] = float(theta)
    for k, i in idx_y.items():
        ub[i] = 1.0
    if ancla is not None:
        lb[idx_y[ancla]] = 1.0

    integrality = np.zeros(n, dtype=int)
    for i in idx_y.values():
        integrality[i] = 1

    A, lower, upper = [], [], []
    # Balance de cada metabolito: producción - consumo + importación - exportación = 0
    for met in metabolitos:
        fila = np.zeros(n)
        for j, r in enumerate(reacciones):
            fila[j] = r["stoich"].get(met, 0.0)
        if met in idx_u:
            fila[idx_u[met]] = 1.0
        if met == producto:
            fila[idx_e] = -1.0
        A.append(fila)
        lower.append(0.0)
        upper.append(0.0)

    # Activar reacción solo si se selecciona su especie
    for j, r in enumerate(reacciones):
        fila = np.zeros(n)
        fila[j] = 1.0
        fila[idx_y[r["especie"]]] = -float(r["capacidad"])
        A.append(fila)
        lower.append(-np.inf)
        upper.append(0.0)

    res = milp(
        c=objetivo,
        integrality=integrality,
        bounds=Bounds(lb, ub),
        constraints=LinearConstraint(np.asarray(A), lower, upper),
        options={"disp": False, "presolve": False},
    )
    if not res.success:
        return {"factible": False, "mensaje": res.message}
    x = res.x
    return {
        "factible": True,
        "especies": [k for k, i in idx_y.items() if x[i] > 0.5],
        "produccion": float(x[idx_e]),
        "captaciones": {m: float(x[i]) for m, i in idx_u.items()},
        "flujos": pd.DataFrame([
            {"especie": r["especie"], "reaccion": r["nombre"],
             "flujo": round(float(x[j]), 6)}
            for j, r in enumerate(reacciones)
        ])
    }


def mostrar(resultado):
    if not resultado["factible"]:
        print("No hay comunidad factible:", resultado["mensaje"])
        return
    print("Comunidad mínima:", ", ".join(resultado["especies"]))
    print("Producción del objetivo:", round(resultado["produccion"], 6))
    print("Captaciones:", {k: round(v, 6) for k, v in resultado["captaciones"].items()})
    display(resultado["flujos"])


## 3. Caso A: levodopa (ilustrativo)

Se representa la cascada conocida con **capacidades arbitrarias de juguete**, no inferidas de AGORA2. Cambiar el nombre de las especies no altera el código del solucionador.

La producción exige cooperación entre una especie que transforma levodopa en dopamina y otra que transforma dopamina en m-tiramina.

In [3]:
ruta_levodopa = [
    {"especie": "E_faecalis", "nombre": "3HLYTCL", "stoich": {"levodopa": -1, "dopamina": 1}, "capacidad": 1.0},
    {"especie": "E_lenta", "nombre": "DOPADH", "stoich": {"dopamina": -1, "m_tiramina": 1}, "capacidad": 1.0},
]

resultado_a = comunidad_minima(
    reacciones=ruta_levodopa,
    medio={"levodopa": 1.0},
    producto="m_tiramina",
    theta=0.8,
)
mostrar(resultado_a)

✅ Comunidad mínima: E_faecalis, E_lenta
Producción del objetivo: 0.8
Captaciones: {'levodopa': 0.8}


,especie,reaccion,flujo
0,E_faecalis,3HLYTCL,0.8
1,E_lenta,DOPADH,0.8


### Control negativo: retirar el sustrato

Al fijar la captación externa de levodopa en cero, la ruta no debería producir m-tiramina.

In [4]:
sin_levodopa = comunidad_minima(
    ruta_levodopa, medio={"levodopa": 0.0}, producto="m_tiramina", theta=0.8
)
print("¿Factible sin levodopa?:", sin_levodopa["factible"])
assert not sin_levodopa["factible"]

¿Factible sin levodopa?: False


## 4. Caso B: otra biotransformación genérica

Ahora cambiamos **todos los nombres de metabolitos y organismos**. Existen dos especies complementarias (`A` y `B`) y una tercera (`C`) que realiza directamente la conversión, pero con capacidad baja.

- Si el mínimo de producción es **0,8**, necesitamos `A + B`.
- Si el mínimo es **0,3**, basta `C`.

Esto demuestra que los datos del caso y el umbral de producción pueden cambiar la comunidad seleccionada sin cambiar el algoritmo.

In [5]:
ruta_generica = [
    {"especie": "A", "nombre": "R1", "stoich": {"S": -1, "I": 1}, "capacidad": 1.0},
    {"especie": "B", "nombre": "R2", "stoich": {"I": -1, "P": 1}, "capacidad": 1.0},
    {"especie": "C", "nombre": "R_directa", "stoich": {"S": -1, "P": 1}, "capacidad": 0.4},
]

for theta in (0.8, 0.3):
    print(f"\nProducción mínima = {theta}")
    resultado = comunidad_minima(ruta_generica, medio={"S": 1.0}, producto="P", theta=theta)
    mostrar(resultado)


Producción mínima = 0.8
✅ Comunidad mínima: A, B
Producción del objetivo: 0.8
Captaciones: {'S': 0.8}


,especie,reaccion,flujo
0,A,R1,0.8
1,B,R2,0.8
2,C,R_directa,0.0



Producción mínima = 0.3
✅ Comunidad mínima: C
Producción del objetivo: 0.3
Captaciones: {'S': 0.3}


,especie,reaccion,flujo
0,A,R1,-0.0
1,B,R2,0.0
2,C,R_directa,0.3


## 5. ¿Cómo conectarlo con el proyecto real?

En la implementación real (`E2/src/comunidad.py` + `E2/src/milp.py`) se reemplazan estas capacidades hipotéticas por reconstrucciones metabólicas completas, con balances por especie, escalamiento por abundancia, crecimiento común, medio comunitario y restricciones de exportación.

**Mensaje para el informe:** la *formulación de selección mínima* puede aplicarse a distintas transformaciones definiendo el conjunto de organismos, las redes metabólicas, el medio y el producto objetivo. **Este notebook solo ilustra generalidad de la idea**, no valida un segundo caso real ni los supuestos biológicos del modelo.

**Limitaciones del juguete:** sin crecimiento, sin SteadyCom, sin cofactores, sin termodinámica, sin abundancias $X_k$, sin reacciones reversibles y sin restricciones nutricionales detalladas. La solución no predice resultados fisiológicos.

**Para una segunda validación científica**, usar otra cascada documentada y reconstrucciones verificadas, después comprobar factibilidad, controles sin sustrato y dependencia de reacciones sink.